# មេរៀន 05 - Agentic RAG


## Setup

This notebook demonstrates the Agentic RAG (Retrieval-Augmented Generation) pattern using the Microsoft Agent Framework.

**Prerequisites:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — your Azure AI Search service endpoint
- `AZURE_SEARCH_API_KEY` — your Azure AI Search API key
- Azure OpenAI deployment configured via environment variables
- Azure CLI authenticated (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Agentic RAG ជាអ្វី?

RAG បែបប្រពៃណីតាមដានដំណើរការ fix: រកឯកសារ, បន្ទាប់មកបង្កើតចម្លើយ។ **Agentic RAG** ធ្វើដំណើរការឆ្ងាយជាងនេះដោយផ្តល់ឱ្យភ្នាក់ងារឯករាជ្យក្នុងការសម្រេចចិត្តថា **ពេលណា** និង **ធ្វើដូចម្តេច** ក្នុងការទាញយកព័ត៌មាន។

ជាមួយ Agentic RAG ភ្នាក់ងារអាច:
- **សម្រេចចិត្ត** ថាតើត្រូវមានការទាញយកព័ត៌មានមុននឹងឆ្លើយសំណួរ
- **ជ្រើសរើស** ប្រភពទិន្នន័យឬឧបករណ៍ដែលត្រូវសួរ
- **វាយតម្លៃ** លទ្ធផលដែលបានយកនិងអនុវត្តការទាញយកបន្ថែមប្រសិនបើការប្រថាដំបូងមិនគ្រប់គ្រាន់
- **បញ្ចូល** ព័ត៌មានពីជំហានទាញយកជាច្រើនទៅជាចម្លើយដែលរលូន

នេះធ្វើឱ្យភ្នាក់ងារជាមួយប្រសើរ និងត្រឹមត្រូវជាងការប្រព្រឹត្តទៅរហូត - ទាញយកបន្ទាប់មកបង្កើតចម្លើយ។ 


## ការបង្កើតឧបករណ៍ស្វែងរក

នៅក្នុង Agentic RAG, ប្រភពទិន្នន័យខាងក្រៅត្រូវបានគេបញ្ចូលជាគ្រឹះមួយដែលហៅថា **tools** ដែលភ្នាក់ងារអាចបញ្ចោញពេលត្រូវការបាន។ វាអនុញ្ញាតឲ្យភ្នាក់ងារព្យាករណ៍ការយកព័ត៌មានជាការសកម្មភាពមួយទៀតដែលវាអាចធ្វើបាន ជំនួសការជាជំហានបាច់បន្លាយ។

ខាងក្រោមនេះយើងកំណត់មូលដ្ឋានចំណេះដឹងទេសចរណ៍ ហើយបង្ហាញវាជាឧបករណ៍មួយដែលភ្នាក់ងារអាចហៅដើម្បីស្វែងរកព័ត៌មានទីកន្លែងដំណើរកម្សាន្ត។


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## ការបង្កើតភ្នាក់ងារ RAG

ឥឡូវនេះយើងបង្កើតភ្នាក់ងារមួយដែលត្រូវបានណែនាំឲ្យ **ទាញយកព័ត៌មានមុនពេលឆ្លើយតបជានិរន្តរ**។ ភ្នាក់ងារនេះប្រើឧបករណ៍ `search_travel_knowledge` ដើម្បីបណ្ដេញចម្លើយរបស់វាដោយផ្អែកលើមូលដ្ឋានចំណេះដឹង មិនមែនរំពឹងទុកលើទិន្នន័យបណ្តុះបណ្តាលរបស់វាផ្ទាល់ទេ។


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## ការទាញយកម្ដងហើយម្ដងទៀត — សំណុំរូបแบบ Maker-Checker

គុណសម្បត្តិសំខាន់មួយនៃ Agentic RAG គឺ **ការទាញយកម្ដងហើយម្ដងទៀត**។ អ្នកប្រតិបត្តិអាចបំពេញការស្វែងរកជាច្រើនជំនាន់ដើម្បីបញ្ជាក់, បញ្ចេញរំលាយ, ឬពង្រឹងលើលទ្ធផលដំណើរការដំបូងរបស់ខ្លួន — ដូចជាការងាររបស់ "maker-checker":

1. **ជំហានអ្នកបង្កើត**: អ្នកប្រតិបត្តិការ ទាញយកព័ត៌មានដំបូង និងរៀបចំការឆ្លើយតប។
2. **ជំហានអ្នកត្រួតពិនិត្យ**: អ្នកប្រតិបត្តិការ បំពេញការទាញយកបន្ថែម ដើម្បីបញ្ជាក់ព័ត៌មាន ឬបំពេញចន្លោះ។

ខាងក្រោមនេះ អ្នកប្រតិបត្តិ ត្រូវបានសុំឆ្លើយសំនួរដែលទាមទារការប្រៀបធៀបគោលដៅច្រើន ដោយបណ្តាបង្ខេបជាការស្វែងរកច្រើនដង។


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## សង្ខេប

នៅក្នុងមេរៀននេះ អ្នកបានរៀនពីរបៀបបង្កើតប្រព័ន្ធ **Agentic RAG** ដោយប្រើប្រាស់ Microsoft Agent Framework៖

- **Agentic RAG** អនុញ្ញាតឲ្យភ្នាក់ងារសម្រេចចិត្តដោយខ្លួនឯងពេលណាដែលត្រូវយកព័ត៌មាន មិនមែនជាការតំណើរការដាក់ជាស្តង់ដាទេ។
- **ឧបករណ៍ជា ប្រភពទិន្នន័យ**៖ មូលដ្ឋានចំណេះដឹងខាងក្រៅ (ដូចជា Azure AI Search) ត្រូវបានវេចខ្ចប់ជាឧបករណ៍ដែលភ្នាក់ងារអាចហៅប្រើបាន។
- **ការយកផ្សេងៗជាដំណើរការ**៖ លំនាំ maker-checker អនុញ្ញាតឲ្យភ្នាក់ងារអាចធ្វើបានច្រើនជុំក្នុងការយកព័ត៌មាន — ស្វែងរក, ផ្ទៀងផ្ទាត់, និងកែលម្អ — មុននឹងផ្តល់ចម្លើយចុងក្រោយ។

ក្នុងប្រតិបត្ដិការផ្លូវការអ្នកនឹងជំនួស `TRAVEL_KNOWLEDGE_BASE` នៅក្នុងម៉ោនម៉ែនឌីងជាមួយសន្ទស្សន៍នៃ Azure AI Search ដែលពិតប្រាកដ ដើម្បីគ្រប់គ្រងការយកឯកសារធំទូលាយទាក់ទងនឹងការធ្វើដំណើរ។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
